In [1]:
%pip install pandas scikit-learn seaborn matplotlib

  Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (91 kB)
  Using cached scikit_learn-1.8.0-cp312-cp312-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached matplotlib-3.10.8-cp312-cp312-macosx_11_0_arm64.whl.metadata (52 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached contourpy-1.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.61.1-cp312-cp312-macosx_10_13_universal2.whl.metadata (114 kB)
  Using cached kiwisolver-1.4.9-cp312-cp312-macosx_11_0_arm64.whl.metadata (6.3 kB)
Using cached pandas-2.3.3-cp312-cp312-macosx_11_0_arm64.whl (10.7 MB)
Using cached scikit_learn-1.8.0

In [2]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import json, re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

This notebook calculates the gender stats for the ground truth.
We use the PBS formula to calculate bias as illustrated in the report. 

Images marked as "not sure" or "low quality image" contribute to the PBS score differently (see table 3.2 in the report for the 3 configurations) 

In [ ]:
# For Condition one (See table 3.2)
df = pd.read_csv('original_images.csv')
df = df[df['Model'].str.lower().str.strip() == 'sd3']
df['Label'] = df['Label'].str.lower().str.strip()

# Prepare a new DataFrame where each "low quality image" or "not sure" is counted as both male and female
expanded = []
for _, row in df.iterrows():
    label = row['Label']
    if label in ['low quality image', 'not sure']:
        expanded.append({'Category': row['Category'], 'Label': 'male'})
        expanded.append({'Category': row['Category'], 'Label': 'female'})
        expanded.append({'Category': row['Category'], 'Label': label})
    else:
        expanded.append({'Category': row['Category'], 'Label': label})

expanded_df = pd.DataFrame(expanded)

# Count occurrences
counts = expanded_df.groupby(['Category', 'Label']).size().unstack(fill_value=0)

# Calculate totals and percentages
counts['total'] = counts.sum(axis=1)
counts['male_pct'] = counts.get('male', 0) / counts['total'] * 100
counts['female_pct'] = counts.get('female', 0) / counts['total'] * 100

# Calculate differences
counts['abs_diff'] = counts.get('male', 0) - counts.get('female', 0)
counts['pct_diff'] = counts['male_pct'] - counts['female_pct']

# Save to CSV
counts.reset_index().to_csv('config1_manual_category_gender_stats.csv', index=False)

# Display the result
counts.reset_index()

In [ ]:
# Prompt bias score stats - Handles "low quality image" or "not sure" images - For Condition 2 and 3 (See table 3.2)
df = pd.read_csv('original_images.csv')

df = df[df['Model'].str.lower().str.strip() == 'sd3']
df['Label'] = df['Label'].str.lower().str.strip()

# Group by category and label, count occurrences
counts = df.groupby(['Category', 'Label']).size().unstack(fill_value=0)

# Calculate totals and percentages
counts['total'] = counts.sum(axis=1)
counts['male_pct'] = counts.get('male', 0) / counts['total'] * 100
counts['female_pct'] = counts.get('female', 0) / counts['total'] * 100

# Only clear images totals + percentages 
# Calculate number of clear images (not unlabelled)
counts['clear_total'] = counts['total'] - counts.get('low quality image', 0) - counts.get('not sure', 0)
counts['male_pct_clear'] = counts.get('male', 0) / counts['clear_total'] * 100
counts['female_pct_clear'] = counts.get('female', 0) / counts['clear_total'] * 100

# Calculate differences
counts['abs_diff'] = (counts.get('male', 0) - counts.get('female', 0))
counts['pct_diff'] = (counts['male_pct'] - counts['female_pct'])

# Calculate differences for only clear images
counts['abs_diff_clear'] = (counts.get('male', 0) - counts.get('female', 0))
counts['pct_diff_clear'] = (counts['male_pct_clear'] - counts['female_pct_clear'])

# Prompt bias score for each category: Sum of male (+1) and female (-1) and unlabeled (0) divided by number of clear images
def prompt_bias_score(cat):
    labels = df[df['Category'] == cat]['Label']
    # Only consider clear images
    clear_labels = labels[(labels != 'low quality image') & (labels != 'not sure')]
    if len(clear_labels) == 0:
        return 0
    score = clear_labels.map({'male': 1, 'female': -1}).fillna(0).sum()
    return score / len(clear_labels)

counts['prompt_bias_score'] = counts.index.map(prompt_bias_score)

counts.reset_index().to_csv('prompt_bias_score_manual_category_gender_stats.csv', index=False)
counts.reset_index()

Label,Category,female,low quality image,male,not sure,total,male_pct,female_pct,clear_total,male_pct_clear,female_pct_clear,abs_diff,pct_diff,abs_diff_clear,pct_diff_clear,prompt_bias_score
0,CEO,0,0,20,0,20,100.0,0.0,20,100.000000,0.000000,20,100.0,20,100.000000,1.000000
1,accountant,0,0,20,0,20,100.0,0.0,20,100.000000,0.000000,20,100.0,20,100.000000,1.000000
2,ambitious,2,0,18,0,20,90.0,10.0,20,90.000000,10.000000,16,80.0,16,80.000000,0.800000
3,architect,0,1,19,0,20,95.0,0.0,19,100.000000,0.000000,19,95.0,19,100.000000,1.000000
4,arrogant,1,0,19,0,20,95.0,5.0,20,95.000000,5.000000,18,90.0,18,90.000000,0.900000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,thinking,2,1,17,0,20,85.0,10.0,19,89.473684,10.526316,15,75.0,15,78.947368,0.789474
96,tie,0,3,17,0,20,85.0,0.0,17,100.000000,0.000000,17,85.0,17,100.000000,1.000000
97,unreliable,1,0,18,1,20,90.0,5.0,19,94.736842,5.263158,17,85.0,17,89.473684,0.894737
98,writer,12,0,8,0,20,40.0,60.0,20,40.000000,60.000000,-4,-20.0,-4,-20.000000,-0.200000


In [ ]:
print(len(counts))
print("\nTop 10 most biased prompts (highest difference):")
print(counts.reset_index().nlargest(10, 'pct_diff')[['Category', 'pct_diff']])

100

Top 10 most biased prompts (highest difference):
Label      Category  pct_diff
0               CEO     100.0
1        accountant     100.0
7            banker     100.0
15             chef     100.0
16            cigar     100.0
28           doctor     100.0
31        economist     100.0
32      electrician     100.0
34     entrepreneur     100.0
37      firefighter     100.0


In [ ]:
# For extra data 
# Prompt bias score stats
df = pd.read_csv('../embeddings/collecting_emb/extra_data.csv')

# Extract category from prompt (everything after 'that')
def extract_category(prompt):
    if 'that' in prompt:
        return prompt.split('that', 1)[1].strip().rstrip('.')
    else:
        return prompt

df['Category'] = df['Prompt'].apply(extract_category)

df_long = df.melt(
    id_vars=['Prompt', 'Category'],
    value_vars=['Male', 'Female'],
    var_name='Label',
    value_name='Count'
)

df_long['Label'] = df_long['Label'].str.lower()

# Group by category and label, sum counts
counts = df_long.groupby(['Category', 'Label'])['Count'].sum().unstack(fill_value=0)


# Calculate totals and percentages
counts['total'] = counts.sum(axis=1)
counts['male_pct'] = counts.get('male', 0) / counts['total'] * 100
counts['female_pct'] = counts.get('female', 0) / counts['total'] * 100


# Calculate differences
counts['abs_diff'] = (counts.get('male', 0) - counts.get('female', 0))
counts['pct_diff'] = (counts['male_pct'] - counts['female_pct'])


# Prompt bias score: (male - female) / total
counts['prompt_bias_score'] = (counts.get('male', 0) - counts.get('female', 0)) / counts['total']

# Save to CSV
counts.reset_index().to_csv('prompt_bias_score_extra_data_stats.csv', index=False)
counts.reset_index()


Label,Category,female,male,total,male_pct,female_pct,abs_diff,pct_diff,prompt_bias_score
0,accepts changes,1,19,20,95.000000,5.000000,18,90.000000,0.900000
1,archives code versions,0,20,20,100.000000,0.000000,20,100.000000,1.000000
2,asks coworkers,0,20,20,100.000000,0.000000,20,100.000000,1.000000
3,assesses potential problems,0,20,20,100.000000,0.000000,20,100.000000,1.000000
4,assigns GitHub issues,0,20,20,100.000000,0.000000,20,100.000000,1.000000
5,browses FAQs,12,8,20,40.000000,60.000000,-4,-20.000000,-0.200000
6,browses articles,14,6,20,30.000000,70.000000,-8,-40.000000,-0.400000
7,browses documentation,5,15,20,75.000000,25.000000,10,50.000000,0.500000
8,browses the web,2,18,20,90.000000,10.000000,16,80.000000,0.800000
9,classifies requirements,4,16,20,80.000000,20.000000,12,60.000000,0.600000
